<a href="https://colab.research.google.com/github/Eelaf-Adam/RAG-PROJECT/blob/main/rag_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-text-splitters
!pip install langchain langchain-community
!pip install sentence-transformers
!pip install faiss-gpu                  # ← Changed from faiss-cpu to faiss-gpu
!pip install transformers
!pip install pypdf
!pip install accelerate
!pip install bitsandbytes               # ← NEW: enables 4-bit quantization
!pip install einops                     # ← NEW: required by some models

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()  # A file picker will appear
# Upload any PDF or text file from your computer

Saving GumArabic_MarketNewsService Issue1_08.pdf to GumArabic_MarketNewsService Issue1_08.pdf
Saving mahasin research Gum Arabic Value chain.pdf to mahasin research Gum Arabic Value chain.pdf
Saving Rapport_Arabische_Gom_uit_Soedan2511.pdf to Rapport_Arabische_Gom_uit_Soedan2511.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFDirectoryLoader("./")
documents = loader.load()

# --- Split into chunks ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Each chunk is ~500 characters
    chunk_overlap=50      # Chunks overlap slightly to avoid losing context
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[4].page_content)

Split into 557 chunks

Sample chunk:
 deaths and the displacement of more than 12 million 
people (BBC 2025; ICG 2025). Famine is severe and 
widespread, and the UN has called it the world’s 
largest humanitarian crisis ( UN News 2025). The war 
pits the Sudanese Armed Forces (SAF), led by Abdel 
Fattah al-Burhan, against the Rapid Support Forces 
(RSF), a powerful paramilitary group headed by 
Mohamed Hamdan Dagalo ‘Hemedti’, whose origins 
lie in the Janjaweed, the Arab militia that committed


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model... (this may take a minute)")


embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",   # ← Upgraded from all-MiniLM-L6-v2
    model_kwargs={"device": "cuda"},         # ← Changed from "cpu" to "cuda"
    encode_kwargs={"normalize_embeddings": True}  # ← NEW: improves retrieval accuracy
)

# Build the vector store from your chunks
print("Building vector store...")
vectorstore = FAISS.from_documents(chunks, embedding_model)

print("Vector store ready!")

Loading embedding model... (this may take a minute)


/tmp/ipython-input-849/615366398.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Building vector store...
Vector store ready!


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,   # No need for 4-bit, model is tiny
    device_map="auto"
)

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.15
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)
print("TinyLlama ready!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:01<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


TinyLlama ready!


/tmp/ipython-input-849/3618964486.py:25: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=llm_pipeline)


In [ ]:
!pip install langchain-classic

In [ ]:
from langchain_classic.prompts import PromptTemplate

prompt_template = """<|system|>
You are a helpful assistant. Answer using only the provided context.
If the answer is not in the context, say "I don't have enough information."</s>
<|user|>
Context:
{context}

Question: {question}</s>
<|assistant|>"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

# Base retriever — fetch more candidates initially
base_retriever = vectorstore.as_retriever(
    search_type="mmr",           # ← Maximum Marginal Relevance: avoids redundant chunks
    search_kwargs={
        "k": 4,                  # ← Return top 5 chunks (was 3)
        "fetch_k": 200            # ← Consider 20 candidates before picking 5
    }
)

# Optional: filter out chunks that aren't relevant enough
embeddings_filter = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.5     # ← Discard chunks below 50% similarity
)

retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

In [ ]:
from langchain_classic.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

In [ ]:
def ask(question):
    print(f"\nQuestion: {question}")
    print("-" * 50)

    result = rag_chain.invoke({"query": question})

    print(f"Answer: {result['result']}")
    print("\nSources used:")
    for i, doc in enumerate(result['source_documents']):
        print(f"  [{i+1}] Page {doc.metadata.get('page', 'N/A')}: {doc.page_content[:150]}...")

In [ ]:
ask("What is tha amount of gum Arabic export value on average over the last 20 years?")


Question: What is tha amount of gum Arabic export value on average over the last 20 years?
--------------------------------------------------


Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: <|system|>
You are a helpful assistant. Answer using only the provided context.
If the answer is not in the context, say "I don't have enough information."</s>
<|user|>
Context:
consumption of Gum Arabic amounted to 
65, 000 tons in the 2003-2006 period. This 
figure, however, is 22 000 tons lower than 
the average exports and re-exports—for 
producer countries—calculated for that 
same period.  
 
This difference can be partly explained by 
the fact that exports by producer countries 
are not correctly reported. It may also be 
explained by the fact that a proportion of 
Gum Arabic gets re-exported under a 
modified identity, such as toiletries,

particularly urgent given the expected substantial 
increase in global demand for gum arabic in the 
coming years, driven by consumer interest in natural 
ingredients, especially in the food and beverage, 
pharmaceutical, and cosmetics industries. 15
15  Coherent Market Insights (2025) values the gum arabic market in 2025 at USD 485 m